## Notebook to calculate ESP_min & ESP_max values

In [ ]:
from ase.io import read, write
from ase.io.cube import read_cube_data, read_cube
import glob
import pandas as pd
import numpy as np
from rdkit import Chem
from pymatgen.core import Molecule, Structure
from pymatgen.io.gaussian import GaussianOutput, GaussianInput
from esp_related_calcs import get_esp_min_max, compute_affinities
import matplotlib.pyplot as plt
# import nglview as nv
plt.rcParams['font.family'] = 'Arial'

In [10]:
%%bash
pwd
ls -ltr

/Users/riteshk/Library/CloudStorage/Box-Box/Research-postdoc/Collaboration-projects/FCether-LMB_KHW/DFT-calcs/ESP
total 180960
-rw-------@ 1 riteshk  staff     1001 Nov 14 20:20 esp_3.com
-rw-------@ 1 riteshk  staff      908 Nov 14 20:20 esp_2.com
-rw-------@ 1 riteshk  staff      814 Nov 14 20:20 esp_1.com
-rw-------@ 1 riteshk  staff      809 Nov 14 20:20 esp_0.com
-rw-------@ 1 riteshk  staff  4453585 Nov 14 20:22 esp_0.log
-rw-------@ 1 riteshk  staff  6682461 Nov 14 20:23 esp_0_density.cube
-rw-------@ 1 riteshk  staff  6682470 Nov 14 20:24 esp_0.cube
-rw-------@ 1 riteshk  staff  4488244 Nov 14 20:24 esp_1.log
-rw-------@ 1 riteshk  staff  6759524 Nov 14 20:24 esp_1_density.cube
-rw-------@ 1 riteshk  staff  6759533 Nov 14 20:25 esp_1.cube
-rw-------@ 1 riteshk  staff  5106357 Nov 14 20:25 esp_2.log
-rw-------@ 1 riteshk  staff  6740529 Nov 14 20:25 esp_2_density.cube
-rw-------@ 1 riteshk  staff  6740538 Nov 14 20:27 esp_2.cube
-rw-------@ 1 riteshk  staff  5480154 Nov 14 20:27

In [ ]:
# data = read('isolated-solv/1st-iter/scf_esp_0.cube', format='cube', index=':')
# # esp_array = data['data'][1]   # [0]=density, [1]=ESP
# # print(esp_array.min(), esp_array.max())
# # atoms
# data[0]
# write('isolated-solv/1st-iter/scf_struc_0.xyz', format='xyz', images=data[0])
# # write('isolated-solv/1st-iter/scf_struc_0.pdb', format='proteindatabank', data=data[0])

### Mesh approach

In [ ]:
def xyz_to_smiles(xyz_file):
    """Convert XYZ file to SMILES using RDKit"""
    try:
        mol = Chem.MolFromXYZFile(xyz_file)
        if mol is not None:
            # Chem.SanitizeMol(mol)
            return Chem.MolToSmiles(mol)
        return None
    except:
        return None

def log_to_smiles(log_file):
    """Convert Gaussian log file to SMILES using RDKit"""
    try:
        out = GaussianOutput(log_file)
        pym_mol = out.final_structure
        pym_mol.to(filename=log_file.replace('.log', '.mol2'), fmt='mol2')
        mol = Chem.MolFromMol2File(log_file.replace('.log', '.mol2'))
        if mol is not None:
            # Chem.SanitizeMol(mol)
            return Chem.MolToSmiles(mol)
        return None
    except:
        return None


#### For molecules calculated on 09-05-2025

In [ ]:
rho_cube_list = glob.glob('esp_*_density.cube')
esp_cube_list = glob.glob('esp_*.cube')
log_list = glob.glob('esp_*.log')

In [ ]:
esp_cube_list = [x for x in esp_cube_list if x not in rho_cube_list]
len(rho_cube_list)
len(esp_cube_list)

6

In [ ]:
# cube_list[0].split('/')[-1].split('_')[2].split('.')[0]
# rho_cube_list, esp_cube_list
# rho_cube_list.sort(key=lambda x: int(x.split('/')[-1].split('_')[2].split('.')[0])); esp_cube_list.sort(key=lambda x: int(x.split('/')[-1].split('_')[2].split('.')[0])); log_list.sort(key=lambda x: int(x.split('/')[-1].split('_')[2].split('.')[0]))
rho_cube_list.sort(key=lambda x: int(x.split('_')[1])); esp_cube_list.sort(key=lambda x: int(x.split('_')[1].split('.')[0])); log_list.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))
rho_cube_list, esp_cube_list, log_list

(['esp_0_density.cube',
  'esp_1_density.cube',
  'esp_2_density.cube',
  'esp_3_density.cube',
  'esp_4_density.cube',
  'esp_5_density.cube'],
 ['esp_0.cube',
  'esp_1.cube',
  'esp_2.cube',
  'esp_3.cube',
  'esp_4.cube',
  'esp_5.cube'],
 ['esp_0.log',
  'esp_1.log',
  'esp_2.log',
  'esp_3.log',
  'esp_4.log',
  'esp_5.log'])

In [6]:
len(esp_cube_list)
# rho_cube_list.remove(esp_cube_list)
rho_cube_list[0].split('_')[1]

'0'

In [7]:
name_list = []
esp_min_eV_list = []; esp_min_kJmol_list = []
esp_max_eV_list = []; esp_max_kJmol_list = []
alpha_eV_list = []; beta_eV_list = []
alpha_kcalmol_list = []; beta_kcalmol_list = []
S_min_bohr2_list = []; S_max_bohr2_list = []
smiles_list = []

for i in range(len(rho_cube_list)):
    rho_cube_file = rho_cube_list[i]
    esp_cube_file = esp_cube_list[i]
    log_file = log_list[i]
    print(f"Processing {rho_cube_file} ...")
    data_esp = read(esp_cube_file, format='cube', index=':')
    xyz_file_name = esp_cube_file.replace('scf_esp', 'scf_struc').split('.')[0] + '.xyz'
    write(xyz_file_name, format='xyz', images=data_esp[0])
    # # esp_min = data_esp[0].flatten().min() * 27.2114 ## convert from au to eV units
    # # esp_max = data_esp[0].flatten().max() * 27.2114 ## convert from au to eV units
    esp_min_eV, esp_max_eV = get_esp_min_max(rho_cube_file, esp_cube_file, iso=1e-3, tol=0.15, units='eV')
    esp_min_kJmol, esp_max_kJmol = get_esp_min_max(rho_cube_file, esp_cube_file, iso=1e-3, tol=0.15, units='kJ/mol')
    print(f"ESP min: {esp_min_eV}, ESP max: {esp_max_eV}")
    res = compute_affinities(rho_cube_file, esp_cube_file, iso=1e-3)
    alpha_eV_list.append(res['alpha_eV']); beta_eV_list.append(res['beta_eV'])
    alpha_kcalmol_list.append(res['alpha_kcal_mol']); beta_kcalmol_list.append(res['beta_kcal_mol'])
    S_min_bohr2_list.append(res['Smin_bohr2']); S_max_bohr2_list.append(res['Smax_bohr2'])
    ind = rho_cube_file.split('/')[-1].split('_')[2].split('.')[0]
    smiles = log_to_smiles(log_file)
    name_list.append(f'first_{ind}')
    esp_min_eV_list.append(esp_min_eV)
    esp_max_eV_list.append(esp_max_eV)
    esp_min_kJmol_list.append(esp_min_kJmol)
    esp_max_kJmol_list.append(esp_max_kJmol)
    smiles_list.append(smiles)

Processing esp_0_density.cube ...
ESP min: -1.6212897526818, ESP max: 1.2682165770546
Processing esp_1_density.cube ...
ESP min: -1.6239972855888, ESP max: 1.528817299638
Processing esp_2_density.cube ...
ESP min: -1.6121684960946001, ESP max: 1.5833026578258
Processing esp_3_density.cube ...
ESP min: -1.6106990812506, ESP max: 1.5873190583994001
Processing esp_4_density.cube ...
ESP min: -1.148973562464, ESP max: 1.1451966220872
Processing esp_5_density.cube ...
ESP min: -1.0032130522164, ESP max: 1.4343965113566


In [ ]:
# xyz_to_smiles(xyz_file_name)
# mol = Structure.from_file(log_list[0], format='gaussian')
# mol = Structure.from_file('isolated-solv/1st-iter/scf_esp_0.out', format='gaussian')
# out = GaussianOutput('isolated-solv/1st-iter/scf_esp_0.log')
# mol = out.final_structure
# mol.to(filename='isolated-solv/1st-iter/scf_esp_0.mol2', fmt='mol2')

In [ ]:
# mol = Chem.MolFromMol2File('isolated-solv/1st-iter/scf_esp_0.mol2', sanitize=False, removeHs=True)
# sm = Chem.MolToSmiles(mol)
# sm

'[H]C([H])([H])OC([H])([H])C([H])([H])OS([O-])([O-])C(F)(F)F'

In [8]:
dict_esp = {
    'name': log_list,
    'smiles': smiles_list,
    'esp_min_eV': esp_min_eV_list,
    'esp_max_eV': esp_max_eV_list,
    'esp_min_kJmol': esp_min_kJmol_list,
    'esp_max_kJmol': esp_max_kJmol_list,
    'alpha_eV': alpha_eV_list,
    'beta_eV': beta_eV_list,
    'alpha_kcalmol': alpha_kcalmol_list,
    'beta_kcalmol': beta_kcalmol_list,
    'S_min_bohr2': S_min_bohr2_list,
    'S_max_bohr2': S_max_bohr2_list
}
df_esp = pd.DataFrame(dict_esp)
df_esp

,name,smiles,esp_min_eV,esp_max_eV,esp_min_kJmol,esp_max_kJmol,alpha_eV,beta_eV,alpha_kcalmol,beta_kcalmol,S_min_bohr2,S_max_bohr2
0,esp_0.log,None,-1.621290,1.268217,-156.430703,122.364316,-1.038842,1.210676,-23.956256,27.918861,255.748184,400.456959
1,esp_1.log,None,-1.623997,1.528817,-156.691940,147.508466,-0.862646,1.150329,-19.893098,26.527213,292.553376,371.809916
2,esp_2.log,None,-1.612168,1.583303,-155.550636,152.765505,-0.869537,1.195425,-20.051997,27.567146,365.232083,406.651928
3,esp_3.log,None,-1.610699,1.587319,-155.408859,153.153029,-0.796147,1.155986,-18.359578,26.657669,415.810142,439.849951
4,esp_4.log,None,-1.148974,1.145197,-110.859112,110.494693,-0.732058,0.916918,-16.881653,21.144640,358.486486,409.035017
5,esp_5.log,None,-1.003213,1.434397,-96.795359,138.398244,-0.856743,1.314823,-19.756956,30.320549,368.008221,418.384448


In [13]:
esp_min, esp_max = get_esp_min_max(rho_cube_file, esp_cube_file, iso=5e-4)
esp_min, esp_max

(np.float64(-1.3344246003312), np.float64(1.515429297726))

In [15]:
pym_mol_list = [GaussianOutput(log).final_structure for log in log_list]

In [16]:
nv.show_pymatgen(pym_mol_list[0])

NGLWidget()

In [30]:
## Per-atom ESP on Smin/Smax surfaces
# DFETHF
res = compute_affinities(rho_cube_list[0], esp_cube_list[0], iso=1e-3)
esp_min_atom_dfethf = res['Vmin_atom_au'] * 27.211386
esp_max_atom_dfethf = res['Vmax_atom_au'] * 27.211386
esp_min_atom_dfethf

array([-0.78208155, -0.19871215, -0.60423538, -0.18714443, -0.63617291,
       -0.08495734, -0.18987455, -0.35559423, -1.10642836, -0.29195818,
        0.        ,  0.        ,  0.        ,  0.        , -0.18226904,
        0.        , -0.17835952, -0.13106186, -0.09272227, -0.06171158])

In [32]:
mol_0 = pym_mol_list[0]
mol_1 = pym_mol_list[1]
mol_2 = pym_mol_list[2]
mol_3 = pym_mol_list[3]
mol_4 = pym_mol_list[4]
mol_5 = pym_mol_list[5]
mol_0

Molecule Summary
Site: F (-2.7664, 1.2600, -0.0205)
Site: C (-2.3882, 0.0017, -0.4278)
Site: F (-3.4669, -0.8248, -0.2010)
Site: C (-1.2131, -0.4947, 0.3856)
Site: O (-0.0953, 0.2956, 0.0398)
Site: C (1.0844, 0.0304, 0.8132)
Site: C (2.0958, 1.1343, 0.5074)
Site: C (2.7449, 0.6373, -0.7847)
Site: O (2.7346, -0.8023, -0.6956)
Site: C (1.8308, -1.2223, 0.3405)
Site: H (-2.2025, 0.0403, -1.5012)
Site: H (-1.0633, -1.5547, 0.1467)
Site: H (-1.4560, -0.4049, 1.4528)
Site: H (0.8164, -0.0243, 1.8736)
Site: H (1.6167, 2.1084, 0.4079)
Site: H (2.8301, 1.1838, 1.3153)
Site: H (2.1717, 0.9489, -1.6646)
Site: H (3.7793, 0.9652, -0.9034)
Site: H (1.1614, -1.9877, -0.0597)
Site: H (2.4035, -1.6588, 1.1667)

In [22]:
# TFETHF
res = compute_affinities(rho_cube_list[1], esp_cube_list[1], iso=1e-3)
esp_min_atom_tfethf = res['Vmin_atom_au'] * 27.211386
esp_max_atom_tfethf = res['Vmax_atom_au'] * 27.211386
esp_min_atom_tfethf

array([-0.33462733, -0.09857323, -0.35576957, -0.55115162, -0.11757457,
       -0.68997191, -0.057849  , -0.22711469, -0.36949105, -1.11185852,
       -0.29322413,  0.        ,  0.        ,  0.        ,  0.        ,
       -0.17345456, -0.1348035 , -0.20913213, -0.05726715, -0.09006893])

In [33]:
mol_1

Molecule Summary
Site: F (2.0348, 0.3937, -1.2546)
Site: C (2.1818, -0.0039, 0.0321)
Site: F (3.2577, -0.8254, 0.0758)
Site: F (2.4900, 1.0982, 0.7475)
Site: C (0.9473, -0.7176, 0.5698)
Site: O (-0.1763, 0.1276, 0.5706)
Site: C (-1.0614, 0.0153, -0.5654)
Site: C (-2.0415, 1.1838, -0.4895)
Site: C (-3.0850, 0.6761, 0.5083)
Site: O (-3.1226, -0.7572, 0.3407)
Site: C (-1.9984, -1.1883, -0.4423)
Site: H (1.1671, -1.0077, 1.5994)
Site: H (0.8034, -1.6267, -0.0222)
Site: H (-0.4814, 0.0023, -1.4905)
Site: H (-2.4892, 1.3397, -1.4741)
Site: H (-1.5534, 2.1066, -0.1765)
Site: H (-4.0855, 1.0715, 0.3225)
Site: H (-2.7988, 0.9074, 1.5395)
Site: H (-2.3408, -1.5023, -1.4354)
Site: H (-1.5302, -2.0439, 0.0507)

In [23]:
# PFPTHF
res = compute_affinities(rho_cube_list[2], esp_cube_list[2], iso=1e-3)
esp_min_atom_pfpthf = res['Vmin_atom_au'] * 27.211386
esp_max_atom_pfpthf = res['Vmax_atom_au'] * 27.211386
esp_min_atom_pfpthf

array([-0.15043028,  0.        , -0.12476746, -0.35698744, -0.52494319,
       -0.63805371, -0.56649988, -0.14411381, -0.77850094,  0.        ,
       -0.22402468, -0.37051922, -1.10968452, -0.2983188 ,  0.        ,
        0.        ,  0.        ,  0.        , -0.16660491, -0.12863277,
       -0.22857481, -0.11230033, -0.04959689])

In [34]:
mol_2

Molecule Summary
Site: F (2.6873, 1.5174, 0.5098)
Site: C (2.6968, 0.2906, -0.0375)
Site: F (2.9711, 0.4174, -1.3468)
Site: F (3.6786, -0.4173, 0.5322)
Site: C (1.3232, -0.4062, 0.1648)
Site: F (1.1407, -0.5409, 1.5099)
Site: F (1.4363, -1.6638, -0.3520)
Site: C (0.1636, 0.3450, -0.4727)
Site: O (-1.0079, -0.3846, -0.2089)
Site: C (-2.1823, 0.1198, -0.8705)
Site: C (-3.2845, -0.9242, -0.7003)
Site: C (-3.8427, -0.5852, 0.6813)
Site: O (-3.7343, 0.8473, 0.7949)
Site: C (-2.7793, 1.3412, -0.1592)
Site: H (0.3562, 0.4248, -1.5500)
Site: H (0.1366, 1.3547, -0.0465)
Site: H (-1.9430, 0.3264, -1.9187)
Site: H (-4.0445, -0.7739, -1.4711)
Site: H (-2.8977, -1.9402, -0.7800)
Site: H (-4.8924, -0.8564, 0.8053)
Site: H (-3.2566, -1.0614, 1.4751)
Site: H (-2.0219, 1.9327, 0.3622)
Site: H (-3.2933, 1.9899, -0.8765)

In [24]:
# SFBTHF
res = compute_affinities(rho_cube_list[3], esp_cube_list[3], iso=1e-3)
esp_min_atom_sfbthf = res['Vmin_atom_au'] * 27.211386
esp_max_atom_sfbthf = res['Vmax_atom_au'] * 27.211386
esp_min_atom_sfbthf

array([-0.13798746,  0.        , -0.20450606, -0.05684971, -0.21035264,
       -0.46241966, -0.45126508, -0.26127745, -0.58462324, -0.36215258,
       -0.12836302, -0.68999303,  0.        , -0.18271597, -0.3598444 ,
       -1.09802998, -0.2931652 ,  0.        ,  0.        ,  0.        ,
        0.        , -0.1259336 , -0.21190513, -0.12378493, -0.05947383,
       -0.10422761])

In [35]:
mol_3

Molecule Summary
Site: F (2.9551, -0.4845, -1.3378)
Site: C (2.2747, -0.9775, -0.2956)
Site: F (3.0973, -1.7479, 0.4241)
Site: F (1.2775, -1.7429, -0.7603)
Site: C (1.7205, 0.1697, 0.6030)
Site: F (1.0383, -0.4036, 1.6179)
Site: F (2.7944, 0.8108, 1.1307)
Site: C (0.8213, 1.2421, -0.0973)
Site: F (0.4119, 2.0784, 0.8999)
Site: F (1.6345, 1.9691, -0.9221)
Site: C (-0.3736, 0.7770, -0.9184)
Site: O (-1.2233, 0.0124, -0.1041)
Site: C (-2.3984, -0.4798, -0.7758)
Site: C (-3.0417, -1.5251, 0.1336)
Site: C (-3.8747, -0.6620, 1.0798)
Site: O (-4.3393, 0.4393, 0.2759)
Site: C (-3.4977, 0.5867, -0.8804)
Site: H (-0.8490, 1.6939, -1.2895)
Site: H (-0.0123, 0.2071, -1.7834)
Site: H (-2.1113, -0.8788, -1.7537)
Site: H (-3.6871, -2.1738, -0.4638)
Site: H (-2.2961, -2.1384, 0.6395)
Site: H (-3.2681, -0.2831, 1.9099)
Site: H (-4.7477, -1.1759, 1.4850)
Site: H (-4.1020, 0.4341, -1.7805)
Site: H (-3.0887, 1.6007, -0.9034)

In [39]:
# 2PFETHF
res = compute_affinities(rho_cube_list[4], esp_cube_list[4], iso=1e-3)
esp_min_atom_2pfethf = res['Vmin_atom_au'] * 27.211386
esp_max_atom_2pfethf = res['Vmax_atom_au'] * 27.211386
esp_min_atom_2pfethf

array([-0.41028991, -0.01412008, -0.21297463, -0.21149863, -0.39985754,
       -0.63830939, -0.4404862 , -0.10675135, -0.69018746, -0.13581952,
       -0.1892843 , -0.0594794 , -0.14853786, -0.66119832,  0.        ,
       -0.11780399, -0.01308929,  0.        , -0.19255907,  0.        ,
       -0.15209556,  0.        , -0.04126984])

In [36]:
mol_4

Molecule Summary
Site: F (-3.6428, -0.4081, -0.4348)
Site: C (-2.6371, 0.3535, 0.0138)
Site: F (-2.5348, 1.4295, -0.7841)
Site: F (-2.9474, 0.7764, 1.2511)
Site: C (-1.3019, -0.4366, 0.0320)
Site: F (-1.5071, -1.5481, 0.7975)
Site: F (-1.0805, -0.8727, -1.2457)
Site: C (-0.1101, 0.3665, 0.5510)
Site: O (1.0585, -0.4224, 0.5387)
Site: C (1.9513, -0.1867, -0.5665)
Site: C (3.1515, -1.1021, -0.3813)
Site: C (4.0312, -0.2846, 0.5803)
Site: C (3.7168, 1.1646, 0.1807)
Site: O (2.4547, 1.1263, -0.5433)
Site: H (-0.3234, 0.6556, 1.5821)
Site: H (0.0015, 1.2752, -0.0457)
Site: H (1.4062, -0.3125, -1.5033)
Site: H (3.6431, -1.2350, -1.3481)
Site: H (2.8664, -2.0805, 0.0033)
Site: H (5.0921, -0.5155, 0.4788)
Site: H (3.7385, -0.4731, 1.6145)
Site: H (3.6105, 1.8313, 1.0384)
Site: H (4.4634, 1.5814, -0.4996)

In [26]:
# F5DEE
res = compute_affinities(rho_cube_list[5], esp_cube_list[5], iso=1e-3)
esp_min_atom_f5dee = res['Vmin_atom_au'] * 27.211386
esp_max_atom_f5dee = res['Vmax_atom_au'] * 27.211386
esp_min_atom_f5dee

array([-0.57680895, -0.17787418, -0.74133197, -0.19125118, -0.51609994,
       -0.08979747, -0.10924339, -0.48959525, -0.1131113 , -0.06013392,
       -0.48149573, -0.33274925, -0.33729072,  0.        ,  0.        ,
        0.        ,  0.        , -0.05360133, -0.0565442 , -0.03505333,
        0.        ,  0.        ])

In [37]:
mol_5

Molecule Summary
Site: F (5.3817, -0.3493, 0.6573)
Site: C (4.2107, 0.3358, 0.4213)
Site: F (4.4489, 1.1645, -0.6505)
Site: C (3.1206, -0.6608, 0.0951)
Site: O (1.9086, 0.0538, -0.0228)
Site: C (0.7986, -0.7772, -0.3438)
Site: C (-0.4213, 0.1230, -0.4358)
Site: O (-1.5321, -0.7241, -0.7520)
Site: C (-2.7384, -0.0644, -1.0413)
Site: C (-3.5982, 0.1508, 0.1995)
Site: F (-3.8865, -1.0076, 0.8294)
Site: F (-3.0085, 0.9570, 1.1141)
Site: F (-4.7748, 0.7280, -0.1409)
Site: H (3.9966, 0.9631, 1.2868)
Site: H (3.0732, -1.3971, 0.9078)
Site: H (3.3785, -1.1788, -0.8381)
Site: H (0.9653, -1.2880, -1.3003)
Site: H (0.6497, -1.5365, 0.4344)
Site: H (-0.2824, 0.8694, -1.2264)
Site: H (-0.5783, 0.6407, 0.5149)
Site: H (-2.5800, 0.9103, -1.5148)
Site: H (-3.3142, -0.6967, -1.7199)

In [40]:
esp_max_atom_f5dee

array([0.22139781, 0.58899861, 0.08257695, 0.73949936, 0.19312716,
       0.55406957, 0.52437559, 0.19354419, 0.85185412, 0.53635097,
       0.2501981 , 0.27481193, 0.33878422, 0.89075887, 0.99669067,
       0.7658064 , 0.74186904, 0.70640293, 0.79510022, 0.4368636 ,
       1.22664937, 0.98847745])

In [14]:
df_esp.to_csv('esp_data_100625.csv', index=False)

In [5]:
blue = (0, 0.576, 0.902) # 0, 147, 230
green = (0.349,0.745,0.306) # 89, 190, 78
red = (0.984, 0.262, 0.219) # 251, 67, 56 
orange = (0.984, 0.713, 0.305) # 251, 182, 78 
purple = (0.839, 0.286, 0.604) # 214, 73, 1541
anvil = (0.298, 0.78, 0.77) # 76, 199, 196
dark_purple = (0.557, 0, 0.998) # 142, 0, 252
pink = (0.95, 0.78, 0.996) # 242, 199, 154
gray = (0.463,0.463,0.463) # 118, 118, 118